In [1]:
# Add module to path
import os
import sys
from itertools import product
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
import pandas as pd
from tqdm import tqdm
from itertools import product
from numba import jit, njit, types
from numba.typed import Dict
import time

# from darts.dartboards import generate_dartboard
from darts.mdp import SinglePlayerContinuousMDP
from darts.stats import expected_score

In [3]:
from darts.dartboards import DARTBOARD_CONSTANTS

In [4]:
board_size = 128
sigma = 10

mm_per_pixel = 2*DARTBOARD_CONSTANTS['DARTBOARD_RADIUS_MM']/board_size
sigma_pxl = sigma / mm_per_pixel

Sigma = sigma_pxl*sigma_pxl*np.array([[1, 0], [0, 1]])
margin = 0.25*sigma_pxl
game_start = 5

In [5]:
mdp = SinglePlayerContinuousMDP(board_size, Sigma, margin, game_start, point_stride=1)

In [6]:
_ = mdp.probs

100%|██████████| 7441/7441 [00:18<00:00, 393.80it/s] 


In [7]:
values = {
    (score, turn, start): 0 for score, turn, start in product(range(game_start+1), [1, 2, 3], range(2, game_start+1))
    if (start >= score) and (turn!=1 or score==start or score==0)
}

In [8]:
len(values)

44

In [9]:
delta = 1e20
threshold = 0.1

while delta > threshold:
    max_q = -1e20
    
    for state in values:
        state_score, state_turn, state_start = state 
        if state_score == 0 or state_score == 1:
            continue
        
        for point in mdp.points:
            key = tuple(point)
            p = mdp.probs['probs'][key]
            cp = mdp.probs['checkout_probs'][key]
            q = 0
            
            for score in p:
                
                # Valid throw
                if score <= state_score - 2:
                    if state_turn == 1:
                        q += p[score]*(values[(state_score - score, 2, state_start)])
                    elif state_turn == 2:
                        q += p[score]*(values[(state_score - score, 3, state_start)])
                    elif state_turn == 3:
                        q += p[score]*(values[(state_score - score, 1, state_score - score)] - 1)
                    else:
                        raise ValueError('!!!')

                # Checkout
                # Found the problem - point is the aim point, not the scoring point
                elif score == state_score:
                    q += cp[score]*(values[(0, state_turn, state_start)])
                    q += (p[score] - cp[score])*(values[(state_start, 1, state_start)] - 1)

                # Bust
                else:
                    q += p[score]*(values[(state_start, 1, state_start)] - 1)

            if q >= max_q:
                max_q = q
        
        delta = max(delta, np.abs(max_q - values[state]))
        print(state, max_q)
        values[state] = max_q

c:\Users\mikey\anaconda3\envs\darts\Lib\site-packages\numba\typed\typeddict.py:39: NumbaTypeSafetyWarning: unsafe cast from int64 to int32. Precision may be lost.
  return d[key]


(2, 1, 2) -0.17120122691783005
(2, 2, 2) -0.17120122691783005
(2, 2, 3) -0.17120122691783005
(2, 2, 4) -0.17120122691783005
(2, 2, 5) -0.17120122691783005
(2, 3, 2) -0.17120122691783005
(2, 3, 3) -0.17120122691783005
(2, 3, 4) -0.17120122691783005
(2, 3, 5) -0.17120122691783005
(3, 1, 3) -0.17120122691783005
(3, 2, 3) -0.17120122691783005
(3, 2, 4) -0.17120122691783005
(3, 2, 5) -0.17120122691783005
(3, 3, 3) -0.17120122691783005
(3, 3, 4) -0.17120122691783005


KeyboardInterrupt: 